# Threshold curves — `biased_bell_v2` & `biased_bell_v3`

Per-direction logical-error-rate vs `p` curves for the Bell-mediated noise models.
Both v2 and v3 are loaded together; each direction gets one subplot with the two
models drawn as separate curves (colored by model, linestyle by distance).

Each direction is either:

- **bias** `β` → `p_bell_z = β · p` (sweep `p`), or
- **fixed `p_bell_z`** → `p_bell_z` held constant while `p` sweeps.

Reads CSVs written by `tools/decode.py` (preamble `threshold_bell`).
Mirrors `threshold_comparison.ipynb` and `threshold_3d_surface.ipynb` in style.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared analysis backend (also used by tools/plot_thresholds_bell.py) so the
# notebook and the CLI produce identical figures.
from bell_threshold_analysis import (
    load_decode_csvs,
    fit_thresholds,
    plot_direction_curves,
)
# locate_threshold / fit_threshold are only needed for the diagnostic cell below.
from threshold_fit import locate_threshold, fit_threshold

%load_ext autoreload
%autoreload 2

In [ ]:
CSV_GLOB = '../out/collection/threshold_bell,*.csv'
NOISE_MODELS = ['biased_bell_v2', 'biased_bell_v3']  # plotted side by side

# Load + merge repeated runs + unpack metadata (see bell_threshold_analysis).
df = load_decode_csvs(CSV_GLOB, NOISE_MODELS)

print(
    f'{len(df)} rows, '
    f'noise_models={sorted(df["noise_model"].unique())}, '
    f'{df["direction"].nunique()} directions, '
    f'distances={sorted(df["expected_distance"].dropna().unique())}'
)
print('rows per (noise_model, direction):')
print(df.groupby(['noise_model', 'direction']).size().unstack(fill_value=0))

In [ ]:
# Fit a threshold per (noise_model, direction). Same locate -> quadratic-FSS
# pipeline as threshold_3d_surface.ipynb, now shared via bell_threshold_analysis.
# LOGICAL_CUTOFF=0.3 drops saturated points; N_PER_SIDE=2 -> 5-point window.
fits_df = fit_thresholds(df, n_per_side=2, logical_cutoff=0.3, verbose=True)
fits_df

In [ ]:
# Diagnostic plot for a single (noise_model, direction): re-run the fit, then plot
# - data points (per distance, with binomial errorbars)
# - the fitted FSS curves
# - the fit window (shaded) and p_th (vertical line)
#
# Useful when a fit looks off (e.g. bias=30) and you want to see whether
# it's the window, the data, or the model. This cell intentionally uses the
# low-level threshold_fit primitives directly for a per-direction close-up.

DIAG_NOISE_MODEL = 'biased_bell_v3'  # change to inspect v2 or v3
DIAG_DIRECTION = 'bias=30'           # change to inspect another direction
DIAG_N_PER_SIDE = 2
DIAG_LOGICAL_CUTOFF = 0.3

grp = df[(df['noise_model'] == DIAG_NOISE_MODEL) & (df['direction'] == DIAG_DIRECTION)]
if grp.empty:
    raise ValueError(
        f'No rows for ({DIAG_NOISE_MODEL!r}, {DIAG_DIRECTION!r}). '
        f'Available directions for {DIAG_NOISE_MODEL!r}: '
        f'{sorted(df[df["noise_model"] == DIAG_NOISE_MODEL]["direction"].unique())}'
    )

p_arr = grp['p'].to_numpy(dtype=float)
d_arr = grp['expected_distance'].to_numpy(dtype=float)
rounds_arr = grp['rounds'].to_numpy(dtype=float)
k_arr = grp['k'].to_numpy(dtype=float)
shots_arr = grp['shots'].to_numpy(dtype=float)
errors_arr = grp['errors'].to_numpy(dtype=float)

valid = (
    (shots_arr > 0)
    & (errors_arr > 0)
    & (errors_arr < shots_arr * DIAG_LOGICAL_CUTOFF)
)
pL_total = np.where(valid, errors_arr / np.maximum(shots_arr, 1), np.nan)
sigma_total = np.where(
    valid,
    np.sqrt(np.clip(pL_total * (1 - pL_total), 0, None) / np.maximum(shots_arr, 1)),
    np.nan,
)
scale = d_arr / (rounds_arr * np.maximum(k_arr, 1))
pL = 1 - (1 - pL_total) ** scale
sigma = sigma_total * scale * (1 - pL_total) ** (scale - 1)

p_init = locate_threshold(p_arr, d_arr, pL)
fit = fit_threshold(p_arr, d_arr, pL, sigma=sigma, p_init=p_init, n_per_side=DIAG_N_PER_SIDE)

print(f'p_init  = {p_init:.4g}')
print(f'p_th    = {fit.p_th:.4g} +/- {fit.p_th_err:.2g}')
print(f'mu      = {fit.mu:.3g} +/- {fit.mu_err:.2g}')
print(f'window  = {fit.p_window}')
print(f'n_points (in fit) = {fit.n_points}')

fig, ax = plt.subplots(figsize=(8, 6))

distances = sorted(np.unique(d_arr[np.isfinite(d_arr)]))
cmap = plt.get_cmap('viridis')
colors = {d: cmap(i / max(1, len(distances) - 1)) for i, d in enumerate(distances)}

# Smooth p grid for the fit overlay.
finite_p = p_arr[(p_arr > 0) & np.isfinite(p_arr)]
p_smooth = np.geomspace(finite_p.min(), finite_p.max(), 200)

for d in distances:
    mask = (d_arr == d) & np.isfinite(pL) & (pL > 0)
    ax.errorbar(
        p_arr[mask], pL[mask], yerr=sigma[mask],
        fmt='o', color=colors[d], label=f'd={int(d)}',
    )
    ax.plot(
        p_smooth, fit.predict(p_smooth, np.full_like(p_smooth, d)),
        color=colors[d], linestyle='-', alpha=0.7,
    )

# Shade the fit window.
w_lo, w_hi = min(fit.p_window), max(fit.p_window)
ax.axvspan(w_lo, w_hi, alpha=0.1, color='gray', label='fit window')
ax.axvline(fit.p_th, color='k', linestyle='--', linewidth=1,
           label=f'$p_{{th}}={fit.p_th:.3g}$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate (p)')
ax.set_ylabel(r'$p_L$ per round, per observable')
ax.set_title(
    f"{DIAG_NOISE_MODEL} / {DIAG_DIRECTION}: "
    f"$p_{{th}}={fit.p_th:.3g} \\pm {fit.p_th_err:.1g}$, "
    f"$\\mu={fit.mu:.2g}$"
)
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Per-direction logical-error-rate curves, with the fitted threshold drawn as a
# vertical dashed line. Both noise models overlaid: color encodes the noise
# model, marker encodes the distance. Shared with tools/plot_thresholds_bell.py.
fig = plot_direction_curves(df, fits_df, noise_models=NOISE_MODELS)
plt.show()

In [12]:
# Persist fits for the threshold-vs-bias notebook.
out_path = Path('../out/collection/threshold_bell_fits.csv')
fits_df.to_csv(out_path, index=False)
print(f'Saved fits -> {out_path}')
fits_df

Saved fits -> ../out/collection/threshold_bell_fits.csv


,noise_model,direction,direction_kind,bias,p_bell_z_fixed,p_th,p_th_err,mu,mu_err,p_init,n_points
0,biased_bell_v2,bias=0,bias,0.0,NaN,0.002418,2.159251e-05,0.910499,0.044097,0.002512,15
1,biased_bell_v2,bias=1,bias,1.0,NaN,0.002047,1.233021e-05,0.971927,0.035426,0.002154,14
2,biased_bell_v2,bias=2,bias,2.0,NaN,0.001701,2.051408e-05,0.756509,0.026000,0.001585,13
3,biased_bell_v2,bias=3,bias,3.0,NaN,0.001486,9.182376e-06,0.954573,0.026916,0.001585,14
4,biased_bell_v2,bias=5,bias,5.0,NaN,0.001196,1.259919e-05,0.815859,0.025822,0.001166,13
5,biased_bell_v2,bias=10,bias,10.0,NaN,0.000795,4.923095e-06,0.880964,0.021115,0.000858,14
6,biased_bell_v2,bias=20,bias,20.0,NaN,0.000492,6.146875e-06,0.713299,0.025259,0.000464,13
7,biased_bell_v2,bias=30,bias,30.0,NaN,0.000348,2.581181e-06,0.791440,0.017868,0.000341,13
8,biased_bell_v2,bias=50,bias,50.0,NaN,0.000220,1.991365e-06,0.884524,0.030904,0.000251,12
9,biased_bell_v2,bias=100,bias,100.0,NaN,0.000116,8.823396e-07,0.963759,0.036348,0.000136,9
